# Grid Search K-Fold Cross-Validation Unificado (MVCNN - Escala de Cinza)
**Projeto:** Classificação de Espécies de Camarão-Rosa (*Farfantepenaeus* / *Penaeus*)  
**Laboratório:** LABEEC  
**Classes:** *P. brasiliensis*, *P. isabelae*, *P. paulensis*, *P. subtilis* (4 espécies)

---

### Por que este notebook foi criado?
No pipeline anterior, os hiperparâmetros foram avaliados de forma **univariada isolada (*One-Factor-at-a-Time*)**:
* A variação de **Pooling** foi executada com valores arbitrários (`batch_size=8`, `lr=0.001`).
* As variações de **Batch Size** e **Learning Rate** foram executadas fixando o modelo em `pooling=max`.

Como o **Pooling**, a **Taxa de Aprendizado** e o **Tamanho do Lote** possuem fortes interações entre si (o Max Pooling zera 2/3 dos gradientes e exige dinâmicas muito diferentes do Avg Pooling), essa estratégia mascarou o colapso do Max Pooling com taxas menores.

### Objetivos deste Notebook:
1. **Grade Fatorial Completa / Grid Search:** Avaliar todas as combinações cruzadas de:
   * **Poolings:** `AdaptiveAvgPool1d(1)` vs. `AdaptiveMaxPool1d(1)`
   * **Batch Sizes:** `8`, `16` (e opcionalmente `4`)
   * **Learning Rates:** `0.001`, `0.0005`, `0.0001`
2. **Validação Cruzada Pareada Estrita:** Todas as configurações são avaliadas nos **exatos mesmos 5 folds** (`StratifiedKFold` com seed fixa).
3. **Registro por Época Completo (Treino e Validação):** Validação executada a cada época para gerar curvas de aprendizado reais e detectar overfitting/underfitting.
4. **Salvamento Automático Consolidado:**
   * `resultados/hiperparametros/grid_search_metrics_cv.csv` (Métricas consolidadas finais)
   * `resultados/hiperparametros/grid_search_lines_cv.csv` (Curvas de treino e validação das 50 épocas)
   * `resultados/hiperparametros/grid_search_predictions_cv.csv` (Predições Out-of-Fold com probabilidades)
5. **Visualização Imediata:** Heatmaps de comparação 2D e curvas de convergência.


## 1. Configurações de Ambiente, Reprodutibilidade e Importações


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import os
import csv
import copy
import random
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from domain.mvcnn_dataset import MVCNNDataset
from domain.mvcnn_model import MVCNNModel

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de execução: {device}")


## 2. Carregamento do Dataset de Imagens Segmentadas em Escala de Cinza


In [ ]:
def listar_diretorio(path):
    return [os.path.join(path, img) for img in os.listdir(path)]

classes_shrimp = ["P. brasiliensis", "P. isabelae", "P. paulensis", "P. subtilis"]
class_to_idx = {cls: i for i, cls in enumerate(classes_shrimp)}

dataset_dir = Path("../dataset_segmentation_cinza/")
views_list_image_paths = []
labels_paths = []

for class_path in dataset_dir.iterdir():
    if class_path.is_dir():
        label = class_path.name
        for specimen_dir in class_path.iterdir():
            if specimen_dir.is_dir() and any(specimen_dir.iterdir()):
                views_list_image_paths.append(listar_diretorio(specimen_dir))
                labels_paths.append(label)

print(f"Total de espécimes carregados: {len(views_list_image_paths)}")
print(f"Total de rótulos: {len(labels_paths)}")
for cls in classes_shrimp:
    count = labels_paths.count(cls)
    print(f"  - {cls}: {count} amostras")


## 3. Transformações de Data Augmentation e Validação


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


## 4. Funções de Treinamento e Validação com Monitoramento Época a Época


In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            _, preds = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs)
            
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    
    return epoch_loss, epoch_acc, f1_macro, all_labels, all_preds, np.array(all_probs)


## 5. Função de Execução de um Fold Completo


In [ ]:
def run_fold(X_train, y_train, X_val, y_val, pooling_fn, batch_size, lr, num_epochs=50, device=device):
    train_dataset = MVCNNDataset(X_train, y_train, classes_shrimp, train_transform)
    val_dataset = MVCNNDataset(X_val, y_val, classes_shrimp, val_test_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=(device.type == "cuda"))
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=(device.type == "cuda"))
    
    # Instancia o modelo com a função de agregação especificada
    model = MVCNNModel(num_classes=len(classes_shrimp), aggregation=pooling_fn).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    epochs_history = []
    best_val_acc = -1.0
    best_results = None
    
    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        vl_loss, vl_acc, vl_f1, vl_labels, vl_preds, vl_probs = validate_epoch(model, val_loader, criterion, device)
        
        epochs_history.append({
            "epoch": epoch,
            "train_loss": tr_loss,
            "train_acc": tr_acc,
            "val_loss": vl_loss,
            "val_acc": vl_acc,
            "val_f1": vl_f1
        })
        
        # Guarda o melhor estado baseado na acurácia de validação
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_results = {
                "loss": vl_loss,
                "acc": vl_acc,
                "f1": vl_f1,
                "labels": vl_labels,
                "preds": vl_preds,
                "probs": vl_probs,
                "best_epoch": epoch
            }
            
    # Se a última época for desejada em vez da melhor, pode-se alternar
    final_metrics = best_results if best_results is not None else {
        "loss": vl_loss, "acc": vl_acc, "f1": vl_f1,
        "labels": vl_labels, "preds": vl_preds, "probs": vl_probs, "best_epoch": num_epochs
    }
    
    return final_metrics, epochs_history


## 6. Configuração da Grade de Hiperparâmetros (*Grid Search*) e Inicialização dos CSVs


In [ ]:
OUTPUT_DIR = "../resultados/hiperparametros"
os.makedirs(OUTPUT_DIR, exist_ok=True)

metrics_csv_path = os.path.join(OUTPUT_DIR, "grid_search_metrics_cv.csv")
lines_csv_path = os.path.join(OUTPUT_DIR, "grid_search_lines_cv.csv")
preds_csv_path = os.path.join(OUTPUT_DIR, "grid_search_predictions_cv.csv")

# Cria cabeçalhos dos arquivos CSV
with open(metrics_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["fold", "pooling", "batch_size", "learning_rate", "loss", "accuracy", "f1", "best_epoch"])

with open(lines_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["fold", "pooling", "batch_size", "learning_rate", "epoch", "train_loss", "train_acc", "val_loss", "val_acc", "val_f1"])

with open(preds_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["fold", "pooling", "batch_size", "learning_rate", "sample", "true_label", "predicted_label", "prob_0", "prob_1", "prob_2", "prob_3"])

# Definição dos valores a serem testados
param_grid = {
    "pooling": [
        ("avg", nn.AdaptiveAvgPool1d(1)),
        ("max", nn.AdaptiveMaxPool1d(1))
    ],
    "batch_size": [8, 16],
    "learning_rate": [0.001, 0.0005, 0.0001]
}

combinations = list(itertools.product(
    param_grid["pooling"],
    param_grid["batch_size"],
    param_grid["learning_rate"]
))

print(f"Total de combinações de hiperparâmetros a avaliar: {len(combinations)}")
for i, ((p_name, _), b_size, lr) in enumerate(combinations, start=1):
    print(f"  [{i:02d}] Pooling: {p_name:<4} | Batch Size: {b_size:<2} | LR: {lr}")


## 7. Execução do Grid Search K-Fold com Registro Contínuo


In [ ]:
N_SPLITS = 5
NUM_EPOCHS = 50

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
splits = list(skf.split(views_list_image_paths, labels_paths))

total_runs = len(combinations) * N_SPLITS
run_count = 0

print(f"Iniciando Grid Search ({total_runs} execuções de folds)...")

for (pool_name, pool_fn), batch_size, lr in combinations:
    print(f"\n{'='*70}")
    print(f"Iniciando Configuração: Pooling={pool_name} | Batch={batch_size} | LR={lr}")
    print(f"{'='*70}")
    
    for fold, (train_idx, val_idx) in enumerate(splits, start=1):
        run_count += 1
        print(f" -> Progresso [{run_count}/{total_runs}]: Fold {fold}/{N_SPLITS}...")
        
        X_train = [views_list_image_paths[i] for i in train_idx]
        y_train = [labels_paths[i] for i in train_idx]
        X_val = [views_list_image_paths[i] for i in val_idx]
        y_val = [labels_paths[i] for i in val_idx]
        
        final_metrics, epochs_history = run_fold(
            X_train, y_train, X_val, y_val,
            pooling_fn=pool_fn,
            batch_size=batch_size,
            lr=lr,
            num_epochs=NUM_EPOCHS,
            device=device
        )
        
        # 1. Salva métricas consolidadas do fold
        with open(metrics_csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                fold, pool_name, batch_size, lr,
                final_metrics["loss"], final_metrics["acc"], final_metrics["f1"], final_metrics["best_epoch"]
            ])
            
        # 2. Salva o histórico de todas as 50 épocas
        with open(lines_csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            for ep_data in epochs_history:
                writer.writerow([
                    fold, pool_name, batch_size, lr,
                    ep_data["epoch"],
                    ep_data["train_loss"], ep_data["train_acc"],
                    ep_data["val_loss"], ep_data["val_acc"], ep_data["val_f1"]
                ])
                
        # 3. Salva predições e probabilidades Out-of-Fold
        with open(preds_csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            for sample_idx, (t_lbl, p_lbl, probs) in enumerate(zip(final_metrics["labels"], final_metrics["preds"], final_metrics["probs"])):
                writer.writerow([
                    fold, pool_name, batch_size, lr,
                    sample_idx, t_lbl, p_lbl,
                    probs[0], probs[1], probs[2], probs[3]
                ])

print(f"\nGrid Search Cross-Validation CONCLUÍDO com sucesso!")
print(f"Resultados persistidos em: {OUTPUT_DIR}")


## 8. Análise Imediata dos Resultados: Tabela Resumo e Heatmap Interativo


In [ ]:
df_metrics = pd.read_csv(metrics_csv_path)

# Tabela consolidada com média e desvio padrão
summary = df_metrics.groupby(["pooling", "batch_size", "learning_rate"]).agg({
    "accuracy": ["mean", "std"],
    "f1": ["mean", "std"],
    "loss": ["mean", "std"]
}).reset_index()

# Ordena pela acurácia média decrescente
summary = summary.sort_values(("accuracy", "mean"), ascending=False)
display(summary.style.format({
    ("accuracy", "mean"): "{:.2f}%", ("accuracy", "std"): "{:.2f}",
    ("f1", "mean"): "{:.4f}", ("f1", "std"): "{:.4f}",
    ("loss", "mean"): "{:.4f}", ("loss", "std"): "{:.4f}"
}).set_caption("Ranking das Configurações do Grid Search"))


In [ ]:
# Heatmaps comparativos de Acurácia: Avg Pooling vs. Max Pooling
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
fig.suptitle("Comparação de Acurácia Média (%): Avg Pooling vs. Max Pooling", fontsize=15, y=1.02)

for ax, pool_type in zip(axes, ["avg", "max"]):
    df_pool = df_metrics[df_metrics["pooling"] == pool_type]
    pivot = df_pool.pivot_table(index="learning_rate", columns="batch_size", values="accuracy", aggfunc="mean")
    
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu" if pool_type == "avg" else "OrRd", 
                cbar=True, ax=ax, vmin=40, vmax=100, annot_kws={"size": 13, "weight": "bold"})
    ax.set_title(f"Estratégia: {pool_type.upper()} Pooling", fontsize=13, weight="bold")
    ax.set_xlabel("Batch Size", fontsize=11, weight="semibold")
    ax.set_ylabel("Learning Rate", fontsize=11, weight="semibold")

plt.tight_layout()
plt.show()


## 9. Comparação das Curvas de Aprendizado (Época a Época)


In [ ]:
df_lines = pd.read_csv(lines_csv_path)

# Compara as curvas de aprendizado da configuração ótima com Avg vs. Max Pooling
best_batch = 16
best_lr = 0.0005

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f"Dinâmica Época a Época (Batch={best_batch}, LR={best_lr}): Avg vs. Max Pooling", y=1.02)

for ax, metric, title, ylim in zip(axes, ["val_acc", "val_loss"], ["Acurácia de Validação (%)", "Perda de Validação (Loss)"], [(0, 105), (0, 3.5)]):
    for pool_type, color, style in [("avg", "#2ca02c", "-"), ("max", "#d62728", "--")]:
        sub = df_lines[(df_lines["pooling"] == pool_type) & (df_lines["batch_size"] == best_batch) & (df_lines["learning_rate"] == best_lr)]
        ep_summary = sub.groupby("epoch")[metric].agg(["mean", "std"]).reset_index()
        
        ax.plot(ep_summary["epoch"], ep_summary["mean"], label=f"{pool_type.upper()} (Média)", color=color, linestyle=style, lw=2.2)
        ax.fill_between(ep_summary["epoch"], ep_summary["mean"] - ep_summary["std"], ep_summary["mean"] + ep_summary["std"], color=color, alpha=0.18)
        
    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlabel("Época")
    ax.set_ylabel(metric)
    ax.set_ylim(ylim)
    ax.legend(loc="best")
    ax.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()
